# 01 · Entrenamiento en Colab (T4)

Cada integrante corre **sus** corridas. Los resultados se escriben en la carpeta compartida de Drive y la ejecución es **reanudable**: si Colab se desconecta, vuelve a ejecutar todo el notebook y las corridas terminadas se omiten.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.

| Integrante | Corridas | Estimado |
|---|---|---|
| `rosa` | E1 semillas 0-1 + E3 semilla 0 | ~2,6 h |
| `yaiza` | E1 semillas 2-3 + E3 semilla 1 | ~2,6 h |
| `daniel` | E1 semilla 4 + E2 completo | ~2,6 h |

In [ ]:
MEMBER = "rosa"  # <- CAMBIA ESTO: "rosa" | "yaiza" | "daniel"

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rosa-SDev/roya-cafe-cbam-vs-deit.git"
SHARED_DIR = Path("/content/drive/MyDrive/IA_ProyectoFinal_RoCoLe")  # carpeta compartida (acceso directo)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = Path("/content/codigo")
    if not REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repositorio:", REPO, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU: cambia el tipo de entorno a T4"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# Datos: copia la caché desde la carpeta compartida (la generó el notebook 00)
DATA_ROOT = Path("/content/data")
RESULTS_ROOT = SHARED_DIR / "results"
if not (DATA_ROOT / "rocole_256.npz").exists():
    shared = SHARED_DIR / "data"
    if (shared / "rocole_256.npz").exists():
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["cp", str(shared / "rocole_256.npz"), str(shared / "metadata.csv"), str(DATA_ROOT)], check=True)
    else:  # plan B: descargar de Mendeley y generar la caché aquí
        subprocess.run([sys.executable, "-m", "src.prepare_data", "--data-root", str(DATA_ROOT), "--download"], check=True)
os.environ["ROCOLE_DATA_ROOT"] = str(DATA_ROOT)
os.environ["ROCOLE_RESULTS_ROOT"] = str(RESULTS_ROOT)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Datos:", DATA_ROOT, "| resultados:", RESULTS_ROOT)

## (Opcional, solo la primera vez) Benchmark de tiempos

Entrena las 6 configuraciones 3 épocas (unos 5 min) y estima minutos por corrida y horas por integrante en esta GPU. Los resultados del benchmark van a una carpeta local, no a Drive.

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    !python -m src.runs --benchmark --results-root /content/bench

## Corridas asignadas

In [ ]:
!python -m src.runs --member {MEMBER} --results-root "{RESULTS_ROOT}"

## Ejecutar

Ejecuta las corridas pendientes en orden (E1 primero). Cada corrida escribe su JSON en `results/runs/<experimento>/` del Drive compartido, los mapas de ejemplo (`*_maps.npz`) y el mejor modelo en `results/checkpoints/` (este último no se sube a git).

Para lanzar solo E1: agrega `--experiment E1` al final del comando.

In [ ]:
!python -m src.runs --member {MEMBER} --execute

## Avance de todo el equipo

In [ ]:
!python -m src.runs --status --results-root "{RESULTS_ROOT}"